# 11.8 模型有看圖還是猜答案？

# 第 11 章：對齊、問答與語言能力保留

前置：第10章介面與第7章SFT。能力實驗需先有「文字屬性→正確答案」基模，以及可分類的視覺encoder；用scripts/prepare_data.py和pretrain_encoders.py準備。下面隨機模型只驗證資料／梯度，所有品質留給讀者訓練後量測。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：轉接頭尺寸合了，還要學資料的意思**

先確認文字基模會用文字屬性回答，再學圖片接頭。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/alignment.svg")))

**先想一想：**隨機打亂圖片後分數沒下降，可能說明什麼？

先留出屬性組合，再用同一道題替換圖片。正確VLM應依新的圖像線索改變答案；只看問題的模型會在平衡資料上暴露弱點。

**把直覺寫成數學：**同時量原圖accuracy、替圖後按規則變化率、shuffle後accuracy。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
truth = ["circle", "square", "circle", "square"]
text_only = ["circle"] * 4
print("平衡組合上的問題-only基準", sum(a == b for a, b in zip(truth, text_only)) / 4)
print("還需評trained model原圖與替圖，不能只檢查logits不同")

**如何讀結果：**0.5是這個明示平衡資料的規則基準，不是實際VLM測量。

**只改一件事：**只把類別分佈改爲90:10，觀察多數猜測的虛高。
